# 311 — Program Robustness

## Objective

Evaluate the robustness of the associations between the eight ICA programs selected in Notebook 310 (`q < 0.05`) and the primary pharmacological phenotype in the 712-model DepMap–GDSC cohort. The primary phenotype is the median raw `LN_IC50` across drugs tested for each model.

## Robustness analyses

- **Lineage dependence.** Compare discovery and lineage-adjusted associations, and use leave-one-lineage-out analyses to assess the influence of individual lineages.
- **Stability and null comparison.** Use lineage-stratified bootstrap resampling to estimate uncertainty and assess directional stability. Within-lineage permutations provide a reference distribution for the adjusted associations.
- **Phenotype and covariate sensitivity.** Compare results across the alternative phenotype representations defined in Notebook 309 and assess sensitivity to drug-response coverage and available cell-line covariates.

## Outputs and interpretation

The notebook exports a candidate-level summary with effect estimates, diagnostic results, and a robustness classification, alongside metadata documenting the analysis settings. The summary provides cell-line robustness evidence for downstream cross-system comparison. Because candidate selection and robustness assessment use the same cohort, the classifications describe internal post-selection robustness.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

PHARMACOLOGY_DIR = Paths.pharmacology
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs

PHENOTYPE_SENSITIVITY_PATH = (
    PHARMACOLOGY_DIR
    / "309_phenotype_sensitivity_representations.parquet"
)

PROGRAM_SCORES_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_scores.parquet"
)

PROGRAM_ASSOCIATIONS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_program_phenotype_associations.csv"
)

DISCOVERY_METADATA_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_program_discovery_metadata.json"
)

OUTPUT_DIR = CELL_LINE_PROGRAM_DIR

In [3]:
# =============================================================================
# Load authoritative robustness inputs
# =============================================================================

phenotype_sensitivity = pd.read_parquet(
    PHENOTYPE_SENSITIVITY_PATH
)

ica_program_scores = pd.read_parquet(
    PROGRAM_SCORES_PATH
)

program_associations = pd.read_csv(
    PROGRAM_ASSOCIATIONS_PATH
)

with DISCOVERY_METADATA_PATH.open("r", encoding="utf-8") as handle:
    discovery_metadata = json.load(handle)

print("Phenotype sensitivity :", phenotype_sensitivity.shape)
print("ICA program scores    :", ica_program_scores.shape)
print("Program associations  :", program_associations.shape)

Phenotype sensitivity : (712, 16)
ICA program scores    : (712, 51)
Program associations  : (50, 24)


In [4]:
# =============================================================================
# Freeze discovery-level candidate programs
# =============================================================================

candidate_programs = (
    program_associations
    .loc[program_associations["q_value"].lt(0.05)]
    .copy()
)

candidate_program_ids = candidate_programs[
    "program_id"
].tolist()

print("Frozen candidate programs:", len(candidate_program_ids))
print(candidate_program_ids)

Frozen candidate programs: 8
['ICA_PROGRAM_10', 'ICA_PROGRAM_15', 'ICA_PROGRAM_17', 'ICA_PROGRAM_18', 'ICA_PROGRAM_33', 'ICA_PROGRAM_36', 'ICA_PROGRAM_37', 'ICA_PROGRAM_47']


In [5]:
# =============================================================================
# Construct candidate-program robustness table
# =============================================================================

phenotype_ids = phenotype_sensitivity["ModelID"]
score_ids = ica_program_scores["ModelID"]

if phenotype_ids.isna().any() or score_ids.isna().any():
    raise ValueError("Missing ModelID values in the Notebook 309 or 310 input.")

if set(phenotype_ids) != set(score_ids):
    raise ValueError("Notebooks 309 and 310 contain different sets of ModelID values.")

if len(phenotype_ids) != discovery_metadata["n_models"]:
    raise ValueError("The cohort size differs from the Notebook 310 metadata.")

if len(candidate_program_ids) != discovery_metadata["phenotype_association"]["n_fdr_candidates"]:
    raise ValueError("The candidate count differs from the Notebook 310 metadata.")

robustness_data = phenotype_sensitivity.merge(
    ica_program_scores[["ModelID", *candidate_program_ids]],
    on="ModelID",
    how="inner",
    validate="one_to_one",
)

print("Robustness analysis table:", robustness_data.shape)

Robustness analysis table: (712, 24)


In [6]:
# =============================================================================
# Quantify lineage dependence of candidate program scores
# =============================================================================

lineage_eta_squared = []

for program_id in candidate_program_ids:
    values = robustness_data[program_id]
    grand_mean = values.mean()

    ss_total = ((values - grand_mean) ** 2).sum()

    ss_between = (
        robustness_data
        .groupby("OncotreeLineage")[program_id]
        .agg(["size", "mean"])
        .assign(
            weighted_ss=lambda x:
                x["size"] * (x["mean"] - grand_mean) ** 2
        )["weighted_ss"]
        .sum()
    )

    lineage_eta_squared.append(
        {
            "program_id": program_id,
            "lineage_eta_squared": ss_between / ss_total,
        }
    )

lineage_eta_squared = (
    pd.DataFrame(lineage_eta_squared)
    .sort_values("lineage_eta_squared", ascending=False)
    .reset_index(drop=True)
)

lineage_eta_squared

,program_id,lineage_eta_squared
0,ICA_PROGRAM_47,0.447108
1,ICA_PROGRAM_15,0.302411
2,ICA_PROGRAM_33,0.272245
3,ICA_PROGRAM_36,0.142574
4,ICA_PROGRAM_10,0.134836
5,ICA_PROGRAM_18,0.110380
6,ICA_PROGRAM_17,0.105568
7,ICA_PROGRAM_37,0.024071


In [7]:
# =============================================================================
# Construct lineage-adjusted candidate program scores
# =============================================================================

lineage_program_means = (
    robustness_data
    .groupby("OncotreeLineage")[candidate_program_ids]
    .transform("mean")
)

for program_id in candidate_program_ids:
    robustness_data[
        f"{program_id}_lineage_residual"
    ] = (
        robustness_data[program_id]
        - lineage_program_means[program_id]
    )

In [8]:
# =============================================================================
# Evaluate primary lineage-adjusted program associations
# =============================================================================

lineage_adjusted_associations = []

for program_id in candidate_program_ids:
    adjusted_rho, _ = spearmanr(
        robustness_data[f"{program_id}_lineage_residual"],
        robustness_data["selected_phenotype_lineage_residual"],
    )

    discovery_rho = candidate_programs.loc[
        candidate_programs["program_id"].eq(program_id),
        "spearman_rho",
    ].iloc[0]

    lineage_adjusted_associations.append(
        {
            "program_id": program_id,
            "discovery_rho": discovery_rho,
            "lineage_adjusted_rho": adjusted_rho,
            "absolute_rho_change": abs(adjusted_rho) - abs(discovery_rho),
            "effect_retention": (
                abs(adjusted_rho) / abs(discovery_rho)
            ),
        }
    )

lineage_adjusted_associations = (
    pd.DataFrame(lineage_adjusted_associations)
    .sort_values(
        "lineage_adjusted_rho",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

lineage_adjusted_associations

,program_id,discovery_rho,lineage_adjusted_rho,absolute_rho_change,effect_retention
0,ICA_PROGRAM_37,0.171233,0.213141,0.041909,1.244746
1,ICA_PROGRAM_33,0.168724,0.189396,0.020672,1.122517
2,ICA_PROGRAM_36,-0.108113,-0.100454,-0.007659,0.929155
3,ICA_PROGRAM_18,0.106951,0.094920,-0.012031,0.887508
4,ICA_PROGRAM_47,-0.108656,-0.073762,-0.034895,0.678853
5,ICA_PROGRAM_17,-0.121238,-0.051705,-0.069533,0.426477
6,ICA_PROGRAM_10,-0.104859,-0.025555,-0.079305,0.243706
7,ICA_PROGRAM_15,-0.164250,-0.009477,-0.154774,0.057697


In [9]:
# =============================================================================
# Compute leave-one-lineage-out associations
# =============================================================================

lolo_associations = []

for lineage in robustness_data["OncotreeLineage"].unique():
    lineage_mask = robustness_data["OncotreeLineage"].ne(lineage)

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            robustness_data.loc[
                lineage_mask,
                f"{program_id}_lineage_residual",
            ],
            robustness_data.loc[
                lineage_mask,
                "selected_phenotype_lineage_residual",
            ],
        )

        lolo_associations.append(
            {
                "program_id": program_id,
                "left_out_lineage": lineage,
                "spearman_rho": rho,
            }
        )

lolo_associations = pd.DataFrame(lolo_associations)

In [10]:
# =============================================================================
# Summarize leave-one-lineage-out association stability
# =============================================================================

lolo_summary_rows = []

for program_id in candidate_program_ids:
    program_lolo = lolo_associations.loc[
        lolo_associations["program_id"].eq(program_id)
    ].copy()

    full_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    program_lolo["absolute_deviation"] = (
        program_lolo["spearman_rho"] - full_rho
    ).abs()

    most_influential = program_lolo.loc[
        program_lolo["absolute_deviation"].idxmax()
    ]

    lolo_summary_rows.append(
        {
            "program_id": program_id,
            "lineage_adjusted_rho": full_rho,
            "lolo_min_rho": program_lolo["spearman_rho"].min(),
            "lolo_max_rho": program_lolo["spearman_rho"].max(),
            "max_absolute_deviation": most_influential["absolute_deviation"],
            "most_influential_lineage": most_influential["left_out_lineage"],
            "direction_preserved_fraction": (
                np.sign(program_lolo["spearman_rho"])
                .eq(np.sign(full_rho))
                .mean()
            ),
        }
    )

lolo_summary = (
    pd.DataFrame(lolo_summary_rows)
    .sort_values("max_absolute_deviation", ascending=False)
    .reset_index(drop=True)
)

lolo_summary

,program_id,lineage_adjusted_rho,lolo_min_rho,lolo_max_rho,max_absolute_deviation,most_influential_lineage,direction_preserved_fraction
0,ICA_PROGRAM_17,-0.051705,-0.110692,-0.030117,0.058987,Lymphoid,1.000000
1,ICA_PROGRAM_15,-0.009477,-0.028031,0.024164,0.033641,Breast,0.925926
2,ICA_PROGRAM_18,0.094920,0.067586,0.111457,0.027335,Bowel,1.000000
3,ICA_PROGRAM_10,-0.025555,-0.042701,-0.002215,0.023340,Lung,1.000000
4,ICA_PROGRAM_37,0.213141,0.190865,0.232459,0.022276,Lung,1.000000
5,ICA_PROGRAM_47,-0.073762,-0.094388,-0.057668,0.020627,Lymphoid,1.000000
6,ICA_PROGRAM_36,-0.100454,-0.114921,-0.082007,0.018446,CNS/Brain,1.000000
7,ICA_PROGRAM_33,0.189396,0.174455,0.198050,0.014941,Ovary/Fallopian Tube,1.000000


In [11]:
# =============================================================================
# Resampling parameters
# =============================================================================

N_BOOTSTRAP = 2000
RANDOM_SEED = 20260813

rng = np.random.default_rng(RANDOM_SEED)

In [12]:
# =============================================================================
# Compute lineage-stratified bootstrap associations
# =============================================================================

bootstrap_columns = [
    "selected_phenotype",
    *candidate_program_ids,
]

lineage_indices = [
    group.index.to_numpy()
    for _, group in robustness_data.groupby(
        "OncotreeLineage",
        sort=False,
    )
]

bootstrap_associations = []

for bootstrap_id in range(N_BOOTSTRAP):
    sampled_index = np.concatenate(
        [
            rng.choice(
                indices,
                size=len(indices),
                replace=True,
            )
            for indices in lineage_indices
        ]
    )

    bootstrap_sample = robustness_data.loc[
        sampled_index,
        ["OncotreeLineage", *bootstrap_columns],
    ].copy()

    bootstrap_residuals = (
        bootstrap_sample[bootstrap_columns]
        - bootstrap_sample
        .groupby("OncotreeLineage")[bootstrap_columns]
        .transform("mean")
    )

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            bootstrap_residuals[program_id],
            bootstrap_residuals["selected_phenotype"],
        )

        bootstrap_associations.append(
            {
                "bootstrap_id": bootstrap_id,
                "program_id": program_id,
                "spearman_rho": rho,
            }
        )

bootstrap_associations = pd.DataFrame(
    bootstrap_associations
)

print(
    "Bootstrap iterations completed:",
    bootstrap_associations["bootstrap_id"].nunique(),
)

Bootstrap iterations completed: 2000


In [13]:
# =============================================================================
# Summarize lineage-stratified bootstrap stability
# =============================================================================

bootstrap_summary = (
    bootstrap_associations
    .groupby("program_id")["spearman_rho"]
    .agg(
        bootstrap_median="median",
        ci_lower=lambda x: x.quantile(0.025),
        ci_upper=lambda x: x.quantile(0.975),
    )
    .reset_index()
    .merge(
        candidate_programs[
            ["program_id", "spearman_rho"]
        ].rename(
            columns={"spearman_rho": "discovery_rho"}
        ),
        on="program_id",
        how="left",
    )
)

direction_preservation = (
    bootstrap_associations
    .merge(
        bootstrap_summary[
            ["program_id", "discovery_rho"]
        ],
        on="program_id",
        how="left",
    )
    .assign(
        direction_preserved=lambda x:
            np.sign(x["spearman_rho"])
            == np.sign(x["discovery_rho"])
    )
    .groupby("program_id")["direction_preserved"]
    .mean()
    .rename("direction_preserved_fraction")
    .reset_index()
)

bootstrap_summary = (
    bootstrap_summary
    .merge(
        direction_preservation,
        on="program_id",
        how="left",
    )
    .sort_values(
        "bootstrap_median",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

bootstrap_summary

,program_id,bootstrap_median,ci_lower,ci_upper,discovery_rho,direction_preserved_fraction
0,ICA_PROGRAM_37,0.211764,0.135450,0.281007,0.171233,1.0000
1,ICA_PROGRAM_33,0.187231,0.112504,0.261305,0.168724,1.0000
2,ICA_PROGRAM_36,-0.102082,-0.180471,-0.023264,-0.108113,0.9940
3,ICA_PROGRAM_18,0.095507,0.023346,0.173230,0.106951,0.9930
4,ICA_PROGRAM_47,-0.072209,-0.146263,0.003771,-0.108656,0.9655
5,ICA_PROGRAM_17,-0.049606,-0.123460,0.024560,-0.121238,0.8955
6,ICA_PROGRAM_10,-0.029884,-0.100140,0.037878,-0.104859,0.8030
7,ICA_PROGRAM_15,-0.011101,-0.079522,0.061273,-0.164250,0.6255


In [14]:
# =============================================================================
# Define prespecified phenotype-sensitivity panels
# =============================================================================

NEAR_PRIMARY_PHENOTYPES = [
    "mean_ln_ic50",
    "mean_z_score",
    "median_z_score",
]

BROADER_PHENOTYPE_SENSITIVITIES = [
    "mean_auc",
    "median_auc",
]

PHENOTYPE_SENSITIVITY_PANEL = (
    NEAR_PRIMARY_PHENOTYPES
    + BROADER_PHENOTYPE_SENSITIVITIES
)

In [15]:
# =============================================================================
# Evaluate alternative-phenotype association sensitivity
# =============================================================================

phenotype_sensitivity_associations = []

for program_id in candidate_program_ids:
    program_residual = f"{program_id}_lineage_residual"

    for phenotype_name in PHENOTYPE_SENSITIVITY_PANEL:
        rho, _ = spearmanr(
            robustness_data[program_residual],
            robustness_data[
                f"{phenotype_name}_lineage_residual"
            ],
        )

        phenotype_sensitivity_associations.append(
            {
                "program_id": program_id,
                "phenotype_representation": phenotype_name,
                "spearman_rho": rho,
            }
        )

phenotype_sensitivity_associations = pd.DataFrame(
    phenotype_sensitivity_associations
)

phenotype_sensitivity_associations.pivot(
    index="program_id",
    columns="phenotype_representation",
    values="spearman_rho",
).loc[candidate_program_ids]

phenotype_representation,mean_auc,mean_ln_ic50,mean_z_score,median_auc,median_z_score
program_id,,,,,
ICA_PROGRAM_10,0.010781,-0.029638,-0.034826,-0.006511,-0.035165
ICA_PROGRAM_15,-0.027235,-0.022306,-0.014678,-0.012512,-0.015877
ICA_PROGRAM_17,-0.004685,-0.046723,-0.049091,-0.015790,-0.048512
ICA_PROGRAM_18,0.056797,0.107656,0.109347,-0.019813,0.102313
ICA_PROGRAM_33,0.103785,0.196641,0.205651,0.042954,0.202854
ICA_PROGRAM_36,-0.049821,-0.101288,-0.100583,-0.001442,-0.106495
ICA_PROGRAM_37,0.161218,0.218671,0.218622,0.077021,0.227946
ICA_PROGRAM_47,-0.023992,-0.058499,-0.052215,-0.022963,-0.054083


In [16]:
# =============================================================================
# Summarize alternative-phenotype association sensitivity
# =============================================================================

phenotype_sensitivity_summary = []

for program_id in candidate_program_ids:
    program_results = phenotype_sensitivity_associations.loc[
        phenotype_sensitivity_associations["program_id"].eq(program_id)
    ]

    discovery_sign = np.sign(
        candidate_programs.loc[
            candidate_programs["program_id"].eq(program_id),
            "spearman_rho",
        ].iloc[0]
    )

    near_primary = program_results.loc[
        program_results["phenotype_representation"].isin(
            NEAR_PRIMARY_PHENOTYPES
        )
    ]

    broader = program_results.loc[
        program_results["phenotype_representation"].isin(
            BROADER_PHENOTYPE_SENSITIVITIES
        )
    ]

    phenotype_sensitivity_summary.append(
        {
            "program_id": program_id,
            "near_primary_direction_preserved_fraction": (
                np.sign(near_primary["spearman_rho"])
                .eq(discovery_sign)
                .mean()
            ),
            "near_primary_min_abs_rho": (
                near_primary["spearman_rho"].abs().min()
            ),
            "broader_direction_preserved_fraction": (
                np.sign(broader["spearman_rho"])
                .eq(discovery_sign)
                .mean()
            ),
        }
    )

phenotype_sensitivity_summary = (
    pd.DataFrame(phenotype_sensitivity_summary)
    .sort_values(
        [
            "near_primary_direction_preserved_fraction",
            "near_primary_min_abs_rho",
        ],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

phenotype_sensitivity_summary

,program_id,near_primary_direction_preserved_fraction,near_primary_min_abs_rho,broader_direction_preserved_fraction
0,ICA_PROGRAM_15,1.0,0.014678,1.0
1,ICA_PROGRAM_10,1.0,0.029638,0.5
2,ICA_PROGRAM_17,1.0,0.046723,1.0
3,ICA_PROGRAM_47,1.0,0.052215,1.0
4,ICA_PROGRAM_36,1.0,0.100583,1.0
5,ICA_PROGRAM_18,1.0,0.102313,0.5
6,ICA_PROGRAM_33,1.0,0.196641,1.0
7,ICA_PROGRAM_37,1.0,0.218622,1.0


In [17]:
# =============================================================================
# Construct lineage-adjusted drug-response coverage
# =============================================================================

robustness_data[
    "drug_coverage_lineage_residual"
] = (
    robustness_data["n_unique_drug_ids"]
    - robustness_data
    .groupby("OncotreeLineage")["n_unique_drug_ids"]
    .transform("mean")
)

In [18]:
# =============================================================================
# Evaluate drug-response coverage sensitivity
# =============================================================================

coverage_adjusted_associations = []

coverage = robustness_data[
    "drug_coverage_lineage_residual"
].to_numpy()

design = np.column_stack(
    [
        np.ones(len(coverage)),
        coverage,
    ]
)

phenotype_residual = robustness_data[
    "selected_phenotype_lineage_residual"
].to_numpy()

phenotype_beta = np.linalg.lstsq(
    design,
    phenotype_residual,
    rcond=None,
)[0]

phenotype_coverage_residual = (
    phenotype_residual
    - design @ phenotype_beta
)

for program_id in candidate_program_ids:
    program_residual = robustness_data[
        f"{program_id}_lineage_residual"
    ].to_numpy()

    program_beta = np.linalg.lstsq(
        design,
        program_residual,
        rcond=None,
    )[0]

    program_coverage_residual = (
        program_residual
        - design @ program_beta
    )

    adjusted_rho, _ = spearmanr(
        program_coverage_residual,
        phenotype_coverage_residual,
    )

    lineage_only_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    coverage_adjusted_associations.append(
        {
            "program_id": program_id,
            "lineage_only_rho": lineage_only_rho,
            "lineage_coverage_adjusted_rho": adjusted_rho,
            "effect_retention": (
                abs(adjusted_rho) / abs(lineage_only_rho)
                if lineage_only_rho != 0
                else np.nan
            ),
        }
    )

coverage_adjusted_associations = (
    pd.DataFrame(coverage_adjusted_associations)
    .sort_values("effect_retention")
    .reset_index(drop=True)
)

coverage_adjusted_associations

,program_id,lineage_only_rho,lineage_coverage_adjusted_rho,effect_retention
0,ICA_PROGRAM_15,-0.009477,-0.001849,0.195069
1,ICA_PROGRAM_47,-0.073762,-0.058907,0.798610
2,ICA_PROGRAM_10,-0.025555,-0.021158,0.827945
3,ICA_PROGRAM_33,0.189396,0.181992,0.960906
4,ICA_PROGRAM_37,0.213141,0.215252,1.009902
5,ICA_PROGRAM_17,-0.051705,-0.052797,1.021126
6,ICA_PROGRAM_18,0.094920,0.099694,1.050290
7,ICA_PROGRAM_36,-0.100454,-0.110575,1.100758


In [19]:
# =============================================================================
# Cell-line metadata input paths
# =============================================================================

MODEL_METADATA_PATH = (
    Paths.depmap
    / "Model.csv"
)

DEFAULT_PROFILE_PATH = (
    Paths.depmap
    / "OmicsDefaultModelProfiles.csv"
)

OMICS_PROFILES_PATH = (
    Paths.depmap
    / "OmicsProfiles.csv"
)

In [20]:
# =============================================================================
# Load cell-line metadata
# =============================================================================

model_metadata = pd.read_csv(MODEL_METADATA_PATH)
default_profile_metadata = pd.read_csv(DEFAULT_PROFILE_PATH)
omics_profile_metadata = pd.read_csv(OMICS_PROFILES_PATH)

In [21]:
# =============================================================================
# Identify prioritized cell-line covariate columns
# =============================================================================

covariate_terms = [
    "sex",
    "primary",
    "metast",
    "age",
    "growth",
    "collection",
    "source",
    "medium",
    "formulation",
    "stranded",
    "modelid",
    "profile",
]

for name, table in {
    "Model.csv": model_metadata,
    "OmicsDefaultModelProfiles.csv": default_profile_metadata,
    "OmicsProfiles.csv": omics_profile_metadata,
}.items():
    matched_columns = [
        column
        for column in table.columns
        if any(
            term in column.lower()
            for term in covariate_terms
        )
    ]

    print(f"\n{name}")
    print(matched_columns)


Model.csv
['ModelID', 'OncotreeLineage', 'OncotreePrimaryDisease', 'Age', 'AgeCategory', 'Sex', 'PrimaryOrMetastasis', 'SampleCollectionSite', 'SourceType', 'SourceDetail', 'Stage', 'GrowthPattern', 'FormulationID', 'SangerModelID']

OmicsDefaultModelProfiles.csv
['ModelID', 'ProfileID', 'ProfileType']

OmicsProfiles.csv
['ProfileID', 'ModelID', 'Stranded', 'Source']


In [22]:
# =============================================================================
# Inspect RNA profile metadata categories
# =============================================================================

print("Default profile types:")
print(
    default_profile_metadata["ProfileType"]
    .value_counts(dropna=False)
)

print("\nRNA strandedness:")
print(
    omics_profile_metadata["Stranded"]
    .value_counts(dropna=False)
)

print("\nProfile sources:")
print(
    omics_profile_metadata["Source"]
    .value_counts(dropna=False)
)

Default profile types:
ProfileType
dna    1929
rna    1673
Name: count, dtype: int64

RNA strandedness:
Stranded
NaN      2828
False    1005
True      685
Name: count, dtype: int64

Profile sources:
Source
BROAD     3492
SANGER    1026
Name: count, dtype: int64


In [23]:
# =============================================================================
# Construct default RNA-profile metadata
# =============================================================================

rna_profile_metadata = (
    default_profile_metadata
    .loc[
        default_profile_metadata["ProfileType"].eq("rna"),
        ["ModelID", "ProfileID"],
    ]
    .merge(
        omics_profile_metadata[
            ["ProfileID", "ModelID", "Stranded", "Source"]
        ],
        on=["ProfileID", "ModelID"],
        how="left",
    )
    .rename(
        columns={
            "Stranded": "RNA_Stranded",
            "Source": "RNA_Source",
        }
    )
)

rna_profile_metadata.head()

,ModelID,ProfileID,RNA_Stranded,RNA_Source
0,ACH-000873,PR-kl468e,False,BROAD
1,ACH-000860,PR-yL4kGI,False,BROAD
2,ACH-001715,PR-pUWMlI,True,BROAD
3,ACH-001627,PR-2WACfU,True,BROAD
4,ACH-000439,PR-tAgQgg,False,BROAD


In [24]:
# =============================================================================
# Construct candidate covariate table
# =============================================================================

candidate_covariates = [
    "Age",
    "Sex",
    "PrimaryOrMetastasis",
    "SampleCollectionSite",
    "SourceType",
    "GrowthPattern",
    "FormulationID",
]

cellline_covariates = (
    robustness_data[["ModelID", "OncotreeLineage"]]
    .merge(
        model_metadata[
            ["ModelID", *candidate_covariates]
        ],
        on="ModelID",
        how="left",
    )
    .merge(
        rna_profile_metadata[
            ["ModelID", "RNA_Stranded", "RNA_Source"]
        ],
        on="ModelID",
        how="left",
    )
)

covariate_inventory = pd.DataFrame(
    {
        "non_missing": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).notna().sum(),
        "coverage_fraction": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).notna().mean(),
        "n_unique": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).nunique(dropna=True),
    }
).sort_values(
    ["coverage_fraction", "n_unique"],
    ascending=[False, True],
)

covariate_inventory

,non_missing,coverage_fraction,n_unique
RNA_Source,712,1.000000,1
RNA_Stranded,712,1.000000,2
Sex,712,1.000000,3
GrowthPattern,712,1.000000,3
SourceType,712,1.000000,11
SampleCollectionSite,712,1.000000,38
FormulationID,678,0.952247,84
PrimaryOrMetastasis,674,0.946629,2
Age,651,0.914326,88


In [25]:
# =============================================================================
# Assess pre-outcome covariate estimability
# =============================================================================

COVARIATE_TYPES = {
    "Sex": "categorical",
    "Age": "continuous",
    "PrimaryOrMetastasis": "categorical",
    "GrowthPattern": "categorical",
    "SampleCollectionSite": "categorical",
    "SourceType": "categorical",
    "FormulationID": "categorical",
    "RNA_Stranded": "categorical",
    "RNA_Source": "categorical",
}

covariate_estimability = []

for covariate, covariate_type in COVARIATE_TYPES.items():
    analysis_data = (
        cellline_covariates[
            ["OncotreeLineage", covariate]
        ]
        .dropna()
        .copy()
    )

    lineage_design = pd.get_dummies(
        analysis_data["OncotreeLineage"],
        drop_first=True,
        dtype=float,
    )

    baseline_design = np.column_stack(
        [
            np.ones(len(analysis_data)),
            lineage_design.to_numpy(),
        ]
    )

    if covariate_type == "continuous":
        covariate_design = (
            analysis_data[[covariate]]
            .astype(float)
            .to_numpy()
        )
    else:
        covariate_design = pd.get_dummies(
            analysis_data[covariate],
            drop_first=True,
            dtype=float,
        ).to_numpy()

    combined_design = np.column_stack(
        [
            baseline_design,
            covariate_design,
        ]
    )

    baseline_rank = np.linalg.matrix_rank(
        baseline_design
    )
    combined_rank = np.linalg.matrix_rank(
        combined_design
    )

    additional_rank = (
        combined_rank - baseline_rank
    )

    covariate_estimability.append(
        {
            "covariate": covariate,
            "coverage_fraction": (
                len(analysis_data)
                / len(cellline_covariates)
            ),
            "additional_rank": additional_rank,
            "design_full_rank": (
                combined_rank
                == combined_design.shape[1]
            ),
            "observations_per_added_parameter": (
                len(analysis_data) / additional_rank
                if additional_rank > 0
                else np.nan
            ),
            "residual_df": (
                len(analysis_data) - combined_rank
            ),
        }
    )

covariate_estimability = pd.DataFrame(
    covariate_estimability
)

covariate_estimability["joint_test_eligible"] = (
    covariate_estimability["coverage_fraction"].ge(0.80)
    & covariate_estimability["additional_rank"].gt(0)
    & covariate_estimability["design_full_rank"]
    & covariate_estimability[
        "observations_per_added_parameter"
    ].ge(10)
    & covariate_estimability["residual_df"].gt(0)
)

covariate_estimability

,covariate,coverage_fraction,additional_rank,design_full_rank,observations_per_added_parameter,residual_df,joint_test_eligible
0,Sex,1.000000,2,True,356.000000,683,True
1,Age,0.914326,1,True,651.000000,623,True
2,PrimaryOrMetastasis,0.946629,1,True,674.000000,646,True
3,GrowthPattern,1.000000,2,True,356.000000,683,True
4,SampleCollectionSite,1.000000,35,False,20.342857,650,False
5,SourceType,1.000000,10,True,71.200000,675,True
6,FormulationID,0.952247,83,True,8.168675,568,False
7,RNA_Stranded,1.000000,1,True,712.000000,684,True
8,RNA_Source,1.000000,0,True,NaN,685,False


In [26]:
# =============================================================================
# Freeze individual and joint covariate sensitivity sets
# =============================================================================

INDIVIDUAL_COVARIATES = [
    "Sex",
    "Age",
    "PrimaryOrMetastasis",
    "GrowthPattern",
    "SampleCollectionSite",
    "SourceType",
    "FormulationID",
    "RNA_Stranded",
]

JOINT_COVARIATES = (
    covariate_estimability
    .loc[
        covariate_estimability["joint_test_eligible"],
        "covariate",
    ]
    .tolist()
)

print("Individual sensitivity covariates:")
print(INDIVIDUAL_COVARIATES)

print("\nJoint-adjustment covariates:")
print(JOINT_COVARIATES)

Individual sensitivity covariates:
['Sex', 'Age', 'PrimaryOrMetastasis', 'GrowthPattern', 'SampleCollectionSite', 'SourceType', 'FormulationID', 'RNA_Stranded']

Joint-adjustment covariates:
['Sex', 'Age', 'PrimaryOrMetastasis', 'GrowthPattern', 'SourceType', 'RNA_Stranded']


In [27]:
# =============================================================================
# Residualization helper
# =============================================================================

def residualize(values, design_matrix):
    design = np.column_stack(
        [
            np.ones(len(design_matrix)),
            np.asarray(design_matrix, dtype=float),
        ]
    )

    coefficients = np.linalg.lstsq(
        design,
        np.asarray(values, dtype=float),
        rcond=None,
    )[0]

    return (
        np.asarray(values, dtype=float)
        - design @ coefficients
    )

In [28]:
# =============================================================================
# Evaluate individual covariate sensitivity
# =============================================================================

covariate_analysis_data = (
    robustness_data
    .merge(
        cellline_covariates.drop(
            columns="OncotreeLineage"
        ),
        on="ModelID",
        how="left",
    )
)

individual_covariate_results = []

for covariate in INDIVIDUAL_COVARIATES:
    covariate_type = COVARIATE_TYPES[covariate]

    for program_id in candidate_program_ids:
        analysis_data = (
            covariate_analysis_data[
                [
                    "OncotreeLineage",
                    "selected_phenotype",
                    program_id,
                    covariate,
                ]
            ]
            .dropna()
            .copy()
        )

        lineage_design = pd.get_dummies(
            analysis_data["OncotreeLineage"],
            drop_first=True,
            dtype=float,
        )

        if covariate_type == "continuous":
            covariate_design = (
                analysis_data[[covariate]]
                .astype(float)
            )
        else:
            covariate_design = pd.get_dummies(
                analysis_data[covariate],
                drop_first=True,
                dtype=float,
            )

        combined_design = pd.concat(
            [
                lineage_design.reset_index(drop=True),
                covariate_design.reset_index(drop=True),
            ],
            axis=1,
        )

        program_lineage_residual = residualize(
            analysis_data[program_id],
            lineage_design,
        )

        phenotype_lineage_residual = residualize(
            analysis_data["selected_phenotype"],
            lineage_design,
        )

        program_adjusted_residual = residualize(
            analysis_data[program_id],
            combined_design,
        )

        phenotype_adjusted_residual = residualize(
            analysis_data["selected_phenotype"],
            combined_design,
        )

        baseline_rho, _ = spearmanr(
            program_lineage_residual,
            phenotype_lineage_residual,
        )

        adjusted_rho, _ = spearmanr(
            program_adjusted_residual,
            phenotype_adjusted_residual,
        )

        program_partial_r2 = (
            1
            - np.sum(program_adjusted_residual ** 2)
            / np.sum(program_lineage_residual ** 2)
        )

        individual_covariate_results.append(
            {
                "program_id": program_id,
                "covariate": covariate,
                "n_models": len(analysis_data),
                "lineage_only_rho": baseline_rho,
                "adjusted_rho": adjusted_rho,
                "effect_retention": (
                    abs(adjusted_rho) / abs(baseline_rho)
                    if baseline_rho != 0
                    else np.nan
                ),
                "program_partial_r2": program_partial_r2,
            }
        )

individual_covariate_results = pd.DataFrame(
    individual_covariate_results
)

print("Individual covariate comparisons:", individual_covariate_results.shape)

Individual covariate comparisons: (64, 7)


In [29]:
# =============================================================================
# Summarize individual covariate sensitivity
# =============================================================================

individual_covariate_results[
    "direction_preserved"
] = (
    np.sign(individual_covariate_results["adjusted_rho"])
    == np.sign(individual_covariate_results["lineage_only_rho"])
)

individual_covariate_summary = []

for program_id in candidate_program_ids:
    program_results = individual_covariate_results.loc[
        individual_covariate_results["program_id"].eq(program_id)
    ].copy()

    most_attenuating = program_results.loc[
        program_results["effect_retention"].idxmin()
    ]

    strongest_covariate = program_results.loc[
        program_results["program_partial_r2"].idxmax()
    ]

    individual_covariate_summary.append(
        {
            "program_id": program_id,
            "min_effect_retention": (
                most_attenuating["effect_retention"]
            ),
            "most_attenuating_covariate": (
                most_attenuating["covariate"]
            ),
            "direction_preserved_fraction": (
                program_results["direction_preserved"].mean()
            ),
            "max_program_partial_r2": (
                strongest_covariate["program_partial_r2"]
            ),
            "strongest_program_covariate": (
                strongest_covariate["covariate"]
            ),
        }
    )

individual_covariate_summary = (
    pd.DataFrame(individual_covariate_summary)
    .sort_values(
        [
            "direction_preserved_fraction",
            "min_effect_retention",
        ]
    )
    .reset_index(drop=True)
)

individual_covariate_summary

,program_id,min_effect_retention,most_attenuating_covariate,direction_preserved_fraction,max_program_partial_r2,strongest_program_covariate
0,ICA_PROGRAM_15,0.137767,Sex,0.75,0.127433,SampleCollectionSite
1,ICA_PROGRAM_17,0.456240,FormulationID,1.00,0.041018,FormulationID
2,ICA_PROGRAM_47,0.611991,FormulationID,1.00,0.137343,FormulationID
3,ICA_PROGRAM_10,0.796341,FormulationID,1.00,0.132691,FormulationID
4,ICA_PROGRAM_18,0.886422,GrowthPattern,1.00,0.106595,FormulationID
5,ICA_PROGRAM_36,0.904643,FormulationID,1.00,0.132628,FormulationID
6,ICA_PROGRAM_33,0.908519,SourceType,1.00,0.120620,FormulationID
7,ICA_PROGRAM_37,0.957973,SampleCollectionSite,1.00,0.107644,FormulationID


In [30]:
# =============================================================================
# Evaluate joint covariate sensitivity
# =============================================================================

joint_analysis_data = (
    covariate_analysis_data[
        [
            "OncotreeLineage",
            "selected_phenotype",
            *candidate_program_ids,
            *JOINT_COVARIATES,
        ]
    ]
    .dropna()
    .copy()
)

lineage_design = pd.get_dummies(
    joint_analysis_data["OncotreeLineage"],
    drop_first=True,
    dtype=float,
)

joint_covariate_blocks = []

for covariate in JOINT_COVARIATES:
    if COVARIATE_TYPES[covariate] == "continuous":
        block = joint_analysis_data[
            [covariate]
        ].astype(float)
    else:
        block = pd.get_dummies(
            joint_analysis_data[covariate],
            prefix=covariate,
            drop_first=True,
            dtype=float,
        )

    joint_covariate_blocks.append(
        block.reset_index(drop=True)
    )

joint_covariate_design = pd.concat(
    joint_covariate_blocks,
    axis=1,
)

combined_design = pd.concat(
    [
        lineage_design.reset_index(drop=True),
        joint_covariate_design,
    ],
    axis=1,
)

phenotype_lineage_residual = residualize(
    joint_analysis_data["selected_phenotype"],
    lineage_design,
)

phenotype_joint_residual = residualize(
    joint_analysis_data["selected_phenotype"],
    combined_design,
)

joint_covariate_results = []

for program_id in candidate_program_ids:
    program_lineage_residual = residualize(
        joint_analysis_data[program_id],
        lineage_design,
    )

    program_joint_residual = residualize(
        joint_analysis_data[program_id],
        combined_design,
    )

    baseline_rho, _ = spearmanr(
        program_lineage_residual,
        phenotype_lineage_residual,
    )

    adjusted_rho, _ = spearmanr(
        program_joint_residual,
        phenotype_joint_residual,
    )

    joint_covariate_results.append(
        {
            "program_id": program_id,
            "n_models": len(joint_analysis_data),
            "lineage_only_rho": baseline_rho,
            "joint_adjusted_rho": adjusted_rho,
            "effect_retention": (
                abs(adjusted_rho) / abs(baseline_rho)
                if baseline_rho != 0
                else np.nan
            ),
            "program_partial_r2": (
                1
                - np.sum(program_joint_residual ** 2)
                / np.sum(program_lineage_residual ** 2)
            ),
        }
    )

joint_covariate_results = (
    pd.DataFrame(joint_covariate_results)
    .sort_values("effect_retention")
    .reset_index(drop=True)
)

joint_covariate_results

,program_id,n_models,lineage_only_rho,joint_adjusted_rho,effect_retention,program_partial_r2
0,ICA_PROGRAM_10,614,-0.026016,-0.020310,0.780663,0.052047
1,ICA_PROGRAM_37,614,0.202498,0.187630,0.926574,0.040786
2,ICA_PROGRAM_33,614,0.214516,0.205076,0.955996,0.057669
3,ICA_PROGRAM_17,614,-0.036104,-0.034682,0.960633,0.038761
4,ICA_PROGRAM_47,614,-0.082972,-0.079950,0.963575,0.029311
5,ICA_PROGRAM_18,614,0.119598,0.124416,1.040286,0.077297
6,ICA_PROGRAM_36,614,-0.091758,-0.102408,1.116065,0.042604
7,ICA_PROGRAM_15,614,-0.010437,-0.014735,1.411822,0.079858


In [31]:
# =============================================================================
# Compute lineage-preserving phenotype permutation nulls
# =============================================================================

N_PERMUTATIONS = 2000
PERMUTATION_SEED = RANDOM_SEED + 1

permutation_rng = np.random.default_rng(
    PERMUTATION_SEED
)

lineage_positions = list(
    robustness_data
    .groupby("OncotreeLineage", sort=False)
    .indices
    .values()
)

phenotype_residual = robustness_data[
    "selected_phenotype_lineage_residual"
].to_numpy()

program_residuals = {
    program_id: robustness_data[
        f"{program_id}_lineage_residual"
    ].to_numpy()
    for program_id in candidate_program_ids
}

permutation_associations = []

for permutation_id in range(N_PERMUTATIONS):
    permuted_phenotype = phenotype_residual.copy()

    for positions in lineage_positions:
        permuted_phenotype[positions] = (
            permutation_rng.permutation(
                permuted_phenotype[positions]
            )
        )

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            program_residuals[program_id],
            permuted_phenotype,
        )

        permutation_associations.append(
            {
                "permutation_id": permutation_id,
                "program_id": program_id,
                "spearman_rho": rho,
            }
        )

permutation_associations = pd.DataFrame(
    permutation_associations
)

print(
    "Lineage-preserving permutations completed:",
    permutation_associations["permutation_id"].nunique(),
)

Lineage-preserving permutations completed: 2000


In [32]:
# =============================================================================
# Summarize lineage-preserving permutation controls
# =============================================================================

permutation_summary = []

for program_id in candidate_program_ids:
    null_rhos = permutation_associations.loc[
        permutation_associations["program_id"].eq(program_id),
        "spearman_rho",
    ]

    observed_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    exceedance_count = (
        null_rhos.abs() >= abs(observed_rho)
    ).sum()

    permutation_summary.append(
        {
            "program_id": program_id,
            "observed_rho": observed_rho,
            "null_median": null_rhos.median(),
            "null_ci_lower": null_rhos.quantile(0.025),
            "null_ci_upper": null_rhos.quantile(0.975),
            "null_abs_95th_percentile": (
                null_rhos.abs().quantile(0.95)
            ),
            "null_exceedance_fraction": (
                (exceedance_count + 1)
                / (N_PERMUTATIONS + 1)
            ),
        }
    )

permutation_summary = (
    pd.DataFrame(permutation_summary)
    .sort_values(
        "observed_rho",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

permutation_summary

,program_id,observed_rho,null_median,null_ci_lower,null_ci_upper,null_abs_95th_percentile,null_exceedance_fraction
0,ICA_PROGRAM_37,0.213141,-0.000570,-0.076630,0.078851,0.076741,0.000500
1,ICA_PROGRAM_33,0.189396,0.000357,-0.073522,0.073177,0.073529,0.000500
2,ICA_PROGRAM_36,-0.100454,-0.001313,-0.077502,0.074999,0.076083,0.011494
3,ICA_PROGRAM_18,0.094920,-0.002320,-0.080710,0.072423,0.076897,0.017991
4,ICA_PROGRAM_47,-0.073762,-0.001251,-0.077091,0.077010,0.077110,0.061969
5,ICA_PROGRAM_17,-0.051705,0.000560,-0.067557,0.068586,0.067610,0.139930
6,ICA_PROGRAM_10,-0.025555,-0.000828,-0.068535,0.066753,0.067544,0.477261
7,ICA_PROGRAM_15,-0.009477,-0.000668,-0.073270,0.066576,0.070284,0.804098


In [33]:
# =============================================================================
# Derive leave-one-lineage-out influence flags
# =============================================================================

lolo_influence_summary = []

for program_id in candidate_program_ids:
    program_lolo = lolo_associations.loc[
        lolo_associations["program_id"].eq(program_id)
    ].copy()

    full_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    sign_reversal = (
        np.sign(program_lolo["spearman_rho"])
        != np.sign(full_rho)
    ).any()

    minimum_effect_retention = (
        program_lolo["spearman_rho"].abs().min()
        / abs(full_rho)
        if full_rho != 0
        else np.nan
    )

    lolo_influence_summary.append(
        {
            "program_id": program_id,
            "minimum_lolo_effect_retention": (
                minimum_effect_retention
            ),
            "lolo_sign_reversal": sign_reversal,
            "single_lineage_influenced": (
                sign_reversal
                or minimum_effect_retention < 0.50
            ),
        }
    )

lolo_influence_summary = (
    pd.DataFrame(lolo_influence_summary)
    .sort_values("minimum_lolo_effect_retention")
    .reset_index(drop=True)
)

lolo_influence_summary

,program_id,minimum_lolo_effect_retention,lolo_sign_reversal,single_lineage_influenced
0,ICA_PROGRAM_10,0.086684,False,True
1,ICA_PROGRAM_15,0.271594,True,True
2,ICA_PROGRAM_17,0.582486,False,False
3,ICA_PROGRAM_18,0.712026,False,False
4,ICA_PROGRAM_47,0.781812,False,False
5,ICA_PROGRAM_36,0.816373,False,False
6,ICA_PROGRAM_37,0.895487,False,False
7,ICA_PROGRAM_33,0.921113,False,False


In [34]:
# =============================================================================
# Integrate prespecified robustness flags
# =============================================================================

robustness_flags = (
    candidate_programs[
        ["program_id", "spearman_rho"]
    ]
    .rename(
        columns={"spearman_rho": "discovery_rho"}
    )
    .merge(
        lineage_adjusted_associations[
            [
                "program_id",
                "lineage_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention": "lineage_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        bootstrap_summary[
            [
                "program_id",
                "direction_preserved_fraction",
            ]
        ].rename(
            columns={
                "direction_preserved_fraction":
                    "bootstrap_direction_preserved_fraction"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        phenotype_sensitivity_summary[
            [
                "program_id",
                "near_primary_direction_preserved_fraction",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        lolo_influence_summary[
            [
                "program_id",
                "single_lineage_influenced",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        coverage_adjusted_associations[
            [
                "program_id",
                "lineage_coverage_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention":
                    "coverage_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        individual_covariate_summary[
            [
                "program_id",
                "min_effect_retention",
                "direction_preserved_fraction",
            ]
        ].rename(
            columns={
                "min_effect_retention":
                    "individual_covariate_min_retention",
                "direction_preserved_fraction":
                    "individual_covariate_direction_fraction",
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        joint_covariate_results[
            [
                "program_id",
                "joint_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention":
                    "joint_covariate_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
)

robustness_flags["lineage_sensitive"] = (
    (
        np.sign(robustness_flags["lineage_adjusted_rho"])
        != np.sign(robustness_flags["discovery_rho"])
    )
    | robustness_flags["lineage_effect_retention"].lt(0.50)
)

robustness_flags["bootstrap_direction_unstable"] = (
    robustness_flags[
        "bootstrap_direction_preserved_fraction"
    ].lt(0.95)
)

robustness_flags["near_primary_phenotype_sensitive"] = (
    robustness_flags[
        "near_primary_direction_preserved_fraction"
    ].lt(1.0)
)

robustness_flags["coverage_sensitive"] = (
    (
        np.sign(
            robustness_flags["lineage_coverage_adjusted_rho"]
        )
        != np.sign(
            robustness_flags["lineage_adjusted_rho"]
        )
    )
    | robustness_flags[
        "coverage_effect_retention"
    ].lt(0.50)
)

robustness_flags["individual_covariate_sensitive"] = (
    robustness_flags[
        "individual_covariate_direction_fraction"
    ].lt(1.0)
    | robustness_flags[
        "individual_covariate_min_retention"
    ].lt(0.50)
)

robustness_flags["joint_covariate_sensitive"] = (
    (
        np.sign(robustness_flags["joint_adjusted_rho"])
        != np.sign(
            joint_covariate_results
            .set_index("program_id")
            .loc[
                robustness_flags["program_id"],
                "lineage_only_rho",
            ]
            .to_numpy()
        )
    )
    | robustness_flags[
        "joint_covariate_effect_retention"
    ].lt(0.50)
)

robustness_flags[
    [
        "program_id",
        "lineage_sensitive",
        "bootstrap_direction_unstable",
        "near_primary_phenotype_sensitive",
        "single_lineage_influenced",
        "coverage_sensitive",
        "individual_covariate_sensitive",
        "joint_covariate_sensitive",
    ]
]

,program_id,lineage_sensitive,bootstrap_direction_unstable,near_primary_phenotype_sensitive,single_lineage_influenced,coverage_sensitive,individual_covariate_sensitive,joint_covariate_sensitive
0,ICA_PROGRAM_10,True,True,False,True,False,False,False
1,ICA_PROGRAM_15,True,True,False,True,True,True,False
2,ICA_PROGRAM_17,True,True,False,False,False,True,False
3,ICA_PROGRAM_18,False,False,False,False,False,False,False
4,ICA_PROGRAM_33,False,False,False,False,False,False,False
5,ICA_PROGRAM_36,False,False,False,False,False,False,False
6,ICA_PROGRAM_37,False,False,False,False,False,False,False
7,ICA_PROGRAM_47,False,False,False,False,False,False,False


In [35]:
# =============================================================================
# Assign hierarchical robustness categories
# =============================================================================

robustness_flags["association_unstable"] = (
    robustness_flags[
        "near_primary_phenotype_sensitive"
    ]
)

robustness_flags["unresolved_confounding"] = (
    ~robustness_flags["association_unstable"]
    & (
        robustness_flags["coverage_sensitive"]
        | robustness_flags[
            "individual_covariate_sensitive"
        ]
        | robustness_flags[
            "joint_covariate_sensitive"
        ]
    )
)

robustness_flags["context_sensitive"] = (
    ~robustness_flags["association_unstable"]
    & ~robustness_flags["unresolved_confounding"]
    & (
        robustness_flags["lineage_sensitive"]
        | robustness_flags[
            "bootstrap_direction_unstable"
        ]
        | robustness_flags[
            "single_lineage_influenced"
        ]
    )
)

robustness_flags["robustness_category"] = np.select(
    [
        robustness_flags["association_unstable"],
        robustness_flags["unresolved_confounding"],
        robustness_flags["context_sensitive"],
    ],
    [
        "ASSOCIATION_UNSTABLE_CANDIDATE",
        "UNRESOLVED_CONFOUNDED_CANDIDATE",
        "CONTEXT_SENSITIVE_CANDIDATE",
    ],
    default="ROBUSTNESS_SUPPORTED_CANDIDATE",
)

robustness_flags[
    [
        "program_id",
        "robustness_category",
    ]
].sort_values(
    ["robustness_category", "program_id"]
)

,program_id,robustness_category
0,ICA_PROGRAM_10,CONTEXT_SENSITIVE_CANDIDATE
3,ICA_PROGRAM_18,ROBUSTNESS_SUPPORTED_CANDIDATE
4,ICA_PROGRAM_33,ROBUSTNESS_SUPPORTED_CANDIDATE
5,ICA_PROGRAM_36,ROBUSTNESS_SUPPORTED_CANDIDATE
6,ICA_PROGRAM_37,ROBUSTNESS_SUPPORTED_CANDIDATE
7,ICA_PROGRAM_47,ROBUSTNESS_SUPPORTED_CANDIDATE
1,ICA_PROGRAM_15,UNRESOLVED_CONFOUNDED_CANDIDATE
2,ICA_PROGRAM_17,UNRESOLVED_CONFOUNDED_CANDIDATE


## Robustness classification

Each candidate receives the first applicable category in the following order:

1. `ASSOCIATION_UNSTABLE_CANDIDATE` — At least one near-primary phenotype fails to preserve the discovery association direction.
2. `UNRESOLVED_CONFOUNDED_CANDIDATE` — Adjustment for drug-response coverage or a cell-line covariate changes the sign or retains less than 50% of the corresponding lineage-only effect.
3. `CONTEXT_SENSITIVE_CANDIDATE` — Lineage adjustment changes the sign or retains less than 50% of the discovery effect, bootstrap direction preservation falls below 95%, or omission of one lineage changes the sign or retains less than 50% of the full-cohort lineage-adjusted effect.
4. `ROBUSTNESS_SUPPORTED_CANDIDATE` — The candidate passes the preceding checks.

In [36]:
# =============================================================================
# Construct integrated candidate robustness summary
# =============================================================================

program_robustness_summary = (
    robustness_flags
    .merge(
        lineage_eta_squared,
        on="program_id",
        how="left",
    )
    .merge(
        lolo_summary[
            [
                "program_id",
                "most_influential_lineage",
                "max_absolute_deviation",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        bootstrap_summary[
            [
                "program_id",
                "bootstrap_median",
                "ci_lower",
                "ci_upper",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        permutation_summary[
            [
                "program_id",
                "null_abs_95th_percentile",
                "null_exceedance_fraction",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        candidate_programs[
            [
                "program_id",
                "cross_method_convergent",
                "program_status",
            ]
        ],
        on="program_id",
        how="left",
    )
)

program_robustness_summary[
    [
        "program_id",
        "robustness_category",
        "discovery_rho",
        "lineage_adjusted_rho",
        "lineage_effect_retention",
        "lineage_eta_squared",
        "bootstrap_direction_preserved_fraction",
        "most_influential_lineage",
        "null_exceedance_fraction",
        "cross_method_convergent",
        "program_status",
    ]
].sort_values(
    [
        "robustness_category",
        "program_id",
    ]
)

,program_id,robustness_category,discovery_rho,lineage_adjusted_rho,lineage_effect_retention,lineage_eta_squared,bootstrap_direction_preserved_fraction,most_influential_lineage,null_exceedance_fraction,cross_method_convergent,program_status
0,ICA_PROGRAM_10,CONTEXT_SENSITIVE_CANDIDATE,-0.104859,-0.025555,0.243706,0.134836,0.8030,Lung,0.477261,True,candidate_with_cross_method_support
3,ICA_PROGRAM_18,ROBUSTNESS_SUPPORTED_CANDIDATE,0.106951,0.094920,0.887508,0.110380,0.9930,Bowel,0.017991,False,candidate_ica_specific
4,ICA_PROGRAM_33,ROBUSTNESS_SUPPORTED_CANDIDATE,0.168724,0.189396,1.122517,0.272245,1.0000,Ovary/Fallopian Tube,0.000500,False,candidate_ica_specific
5,ICA_PROGRAM_36,ROBUSTNESS_SUPPORTED_CANDIDATE,-0.108113,-0.100454,0.929155,0.142574,0.9940,CNS/Brain,0.011494,False,candidate_ica_specific
6,ICA_PROGRAM_37,ROBUSTNESS_SUPPORTED_CANDIDATE,0.171233,0.213141,1.244746,0.024071,1.0000,Lung,0.000500,False,candidate_ica_specific
7,ICA_PROGRAM_47,ROBUSTNESS_SUPPORTED_CANDIDATE,-0.108656,-0.073762,0.678853,0.447108,0.9655,Lymphoid,0.061969,False,candidate_ica_specific
1,ICA_PROGRAM_15,UNRESOLVED_CONFOUNDED_CANDIDATE,-0.164250,-0.009477,0.057697,0.302411,0.6255,Breast,0.804098,True,candidate_with_cross_method_support
2,ICA_PROGRAM_17,UNRESOLVED_CONFOUNDED_CANDIDATE,-0.121238,-0.051705,0.426477,0.105568,0.8955,Lymphoid,0.139930,True,candidate_with_cross_method_support


In [37]:
# =============================================================================
# Robustness output paths
# =============================================================================

ROBUSTNESS_SUMMARY_PATH = (
    OUTPUT_DIR
    / "311_program_robustness_summary.csv"
)

ROBUSTNESS_METADATA_PATH = (
    OUTPUT_DIR
    / "311_program_robustness_metadata.json"
)

In [38]:
# =============================================================================
# Write program-robustness summary
# =============================================================================

program_robustness_summary.to_csv(ROBUSTNESS_SUMMARY_PATH, index=False)
print("Summary written:", project_relative_path(ROBUSTNESS_SUMMARY_PATH))

Summary written: data/processed/cellline_programs/311_program_robustness_summary.csv


In [39]:
# =============================================================================
# Construct program-robustness metadata
# =============================================================================

rna_source_limitation = (
    "RNA-source sensitivity could not be assessed because all "
    f"{len(robustness_data)} models shared one recorded source."
)

robustness_metadata = {
    "notebook": "311_program_robustness",
    "analysis_scope": {
        "candidate_program_count": len(candidate_program_ids),
        "candidate_program_ids": candidate_program_ids,
        "primary_phenotype": "selected_phenotype",
        "cohort_model_count": len(robustness_data),
        "interpretation": (
            "Internal post-selection association-robustness analysis "
            "within the discovery cohort."
        ),
    },
    "resampling": {
        "bootstrap_iterations": N_BOOTSTRAP,
        "bootstrap_seed": RANDOM_SEED,
        "permutation_iterations": N_PERMUTATIONS,
        "permutation_seed": PERMUTATION_SEED,
        "strategy": "lineage-stratified",
    },
    "phenotype_sensitivity": {
        "near_primary_representations": NEAR_PRIMARY_PHENOTYPES,
        "broader_representations": BROADER_PHENOTYPE_SENSITIVITIES,
    },
    "covariates": {
        "individual": INDIVIDUAL_COVARIATES,
        "joint": JOINT_COVARIATES,
        "not_evaluable": {
            "RNA_Source": rna_source_limitation,
            "proliferation": (
                "No frozen upstream cell-line proliferation covariate "
                "was identified; no post hoc expression-derived score "
                "was constructed."
            ),
        },
    },
    "prespecified_thresholds": {
        "minimum_effect_retention": 0.50,
        "minimum_bootstrap_direction_preservation": 0.95,
        "minimum_covariate_coverage_for_joint_test": 0.80,
        "minimum_observations_per_added_parameter": 10,
    },
    "classification_hierarchy": [
        "ASSOCIATION_UNSTABLE_CANDIDATE",
        "UNRESOLVED_CONFOUNDED_CANDIDATE",
        "CONTEXT_SENSITIVE_CANDIDATE",
        "ROBUSTNESS_SUPPORTED_CANDIDATE",
    ],
    "classification_rules": {
        "ASSOCIATION_UNSTABLE_CANDIDATE": (
            "Near-primary phenotype direction is not fully preserved."
        ),
        "UNRESOLVED_CONFOUNDED_CANDIDATE": (
            "The candidate passes the near-primary direction check and "
            "adjustment for drug-response coverage or a cell-line covariate "
            "changes the sign or retains less than 50% of the corresponding "
            "lineage-only effect."
        ),
        "CONTEXT_SENSITIVE_CANDIDATE": (
            "Association is not unstable or confounded but shows "
            "lineage sensitivity, bootstrap directional instability, "
            "or material leave-one-lineage-out influence."
        ),
        "ROBUSTNESS_SUPPORTED_CANDIDATE": (
            "None of the preceding robustness conditions is met."
        ),
    },
    "program_categories": (
        program_robustness_summary.set_index("program_id")["robustness_category"]
        .to_dict()
    ),
    "limitations": [
        (
            "Candidate programs were selected in the same cohort used "
            "for these robustness analyses."
        ),
        (
            "Permutation controls are diagnostic and are not used "
            "as a new candidate-selection procedure."
        ),
        (
            "Drug-response coverage adjustment does not address "
            "drug-family composition or cross-screen robustness."
        ),
        rna_source_limitation,
        (
            "RNA strandedness provides only a limited technical "
            "sensitivity analysis."
        ),
        (
            "Association robustness does not imply causality, clinical "
            "resistance prediction, or therapeutic relevance."
        ),
    ],
}

with ROBUSTNESS_METADATA_PATH.open("w", encoding="utf-8") as handle:
    json.dump(robustness_metadata, handle, indent=2)

print(
    "Metadata written:",
    project_relative_path(ROBUSTNESS_METADATA_PATH),
)

Metadata written: data/processed/cellline_programs/311_program_robustness_metadata.json


## Robustness summary and downstream interpretation

Five of the eight discovery candidates met the internal robustness criteria. All eight preserved their discovery direction across the three near-primary phenotype representations.

| Robustness category | ICA programs |
|---|---|
| `ROBUSTNESS_SUPPORTED_CANDIDATE` | `ICA_PROGRAM_18`, `ICA_PROGRAM_33`, `ICA_PROGRAM_36`, `ICA_PROGRAM_37`, `ICA_PROGRAM_47` |
| `CONTEXT_SENSITIVE_CANDIDATE` | `ICA_PROGRAM_10` |
| `UNRESOLVED_CONFOUNDED_CANDIDATE` | `ICA_PROGRAM_15`, `ICA_PROGRAM_17` |
| `ASSOCIATION_UNSTABLE_CANDIDATE` | None |

`ICA_PROGRAM_37` and `ICA_PROGRAM_33` had the largest absolute lineage-adjusted associations (Spearman ρ = 0.213 and 0.189). `ICA_PROGRAM_47` met the robustness criteria with 96.6% bootstrap direction preservation, while its 95% bootstrap interval included zero.

`ICA_PROGRAM_10` retained 24% of its discovery effect after lineage adjustment and preserved its direction in 80.3% of bootstrap samples. `ICA_PROGRAM_15` had a near-zero lineage-adjusted association (ρ = −0.009) and was sensitive to coverage and individual covariate adjustment. `ICA_PROGRAM_17` retained 46% of its lineage-only effect after individual adjustment for `FormulationID`.

The summary carries all eight candidates and their classifications forward to Phase 4. These classifications describe post-selection robustness within the discovery cohort and provide context for cross-system comparison.

In [ ]:
# =============================================================================
# Verify and register artifacts
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    refresh_notebook_artifacts,
)

output_objects = {
    ROBUSTNESS_SUMMARY_PATH: program_robustness_summary,
    ROBUSTNESS_METADATA_PATH: robustness_metadata,
}
verified_outputs = {}
for path, expected in output_objects.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.suffix == ".json":
        with path.open(encoding="utf-8") as handle:
            observed = json.load(handle)
        if observed != expected:
            raise ValueError(f"Written metadata differs: {path}")
        verified_outputs[path] = None
    else:
        observed = (
            pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        )
        if observed.shape != expected.shape or not observed.columns.equals(
            expected.columns
        ):
            raise ValueError(f"Written table structure differs: {path}")
        verified_outputs[path] = observed.shape

registered_count = refresh_notebook_artifacts(
    "notebooks/phase3_cell_line_discovery_layer/311_program_robustness.ipynb",
    verified_outputs,
    require_unchanged=False,
)
load_artifact_registry()
print(f"Registered {registered_count} verified artifacts.")
